# Download Qwen3.8-27B with Hugging Face Hub

This uses `snapshot_download` with resumable per-file downloads. It does not create a Git clone or duplicate Git LFS metadata.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U 'huggingface_hub[hf_transfer]'

In [ ]:
import os
from getpass import getpass
from pathlib import Path

MODEL_ID = 'Qwen/Qwen3.8-27B'
MODEL_DIR = Path('/content/drive/MyDrive/CrossLLM/models/Qwen3.8-27B')
HF_TOKEN = getpass('Paste Hugging Face token (hidden; press Enter for public access): ')

MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['HF_TOKEN'] = HF_TOKEN
print('Model:', MODEL_ID)
print('Destination:', MODEL_DIR)

In [ ]:
from huggingface_hub import snapshot_download

# Re-run this cell after a disconnect. Existing completed shards are reused.
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=str(MODEL_DIR),
    token=HF_TOKEN or None,
    max_workers=8,
)
print('Download/resume finished:', MODEL_DIR)

In [ ]:
import json
config = MODEL_DIR / 'config.json'
assert config.exists(), 'config.json is missing'
assert json.loads(config.read_text())['model_type'] == 'qwen3_5'
weights = sorted(MODEL_DIR.glob('*.safetensors'))
assert weights, 'No safetensors files found'
pointer_files = [p for p in weights if p.stat().st_size < 1024]
assert not pointer_files, f'Incomplete Hugging Face files: {pointer_files[:3]}'
print(f'Validated {len(weights)} weight shards in {MODEL_DIR}')